In [ ]:
import os
os.environ['OMP_NUM_THREADS'] = '4' #num works
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import openai
import plotly.graph_objects as go
import pandas as pd
import numpy as np
import sys
from data.serialize import SerializerSettings
from models.utils import grid_iter
from models.promptcast import get_promptcast_predictions_data
from models.darts import get_arima_predictions_data
from models.gaussian_process import get_gp_predictions_data
from models.darts import get_TCN_predictions_data, get_NHITS_predictions_data, get_NBEATS_predictions_data
from models.llmtime import get_llmtime_predictions_data
from data.small_context import get_datasets, get_memorization_datasets
from models.validation_likelihood_tuning import get_autotuned_predictions_data
%load_ext autoreload
%autoreload 2


In [ ]:
gpt4_hypers = dict(
    alpha=0.3,
    basic=True,
    temp=1.0,
    top_p=0.8,
    settings=SerializerSettings(base=10, prec=3, signed=True, time_sep=', ', bit_sep='', minus_sign='-')
)

gpt3_hypers = dict(
    temp=.7,
    alpha=0.95,
    beta=0.7,
    basic=[False],
    settings=SerializerSettings(base=10, prec=3, signed=True, half_bin_correction=True)
)

model_hypers = {
    'LLMTime GPT-3.5-turbo': {'model': 'gpt-3.5-turbo', **gpt3_hypers},
    'LLMTime GPT-4': {'model': 'gpt-4', **gpt4_hypers},
    'LLMTime GPT-3': {'model': 'gpt-3.5-turbo-instruct', **gpt3_hypers},
    'GPT2':{'model':'gpt2',**gpt3_hypers},
    'LLMTime GPT-3.5': {'model': 'gpt-3.5', **gpt3_hypers},
    
}

model_predict_fns = {
    #'LLMTime GPT-3.5-turbo': get_llmtime_predictions_data,
    #'LLMTime GPT-3': get_llmtime_predictions_data,
    'LLMTime GPT-4': get_llmtime_predictions_data,
    #'GPT2':get_llmtime_predictions_data
    #'LLMTime GPT-3.5':get_llmtime_predictions_data
}

model_names = list(model_predict_fns.keys())

## 其他LLM

In [ ]:

gemini_hypers = dict(
    alpha=0.95,
    basic=True,
    temp=1.0,
    top_p=0.8,
    settings=SerializerSettings(base=10, prec=3, signed=True, time_sep=', ', bit_sep='', minus_sign='-', half_bin_correction=False)
)

claude_hypers = dict(
    alpha=0.95,
    basic=True,
    temp=1.0,
    top_p=0.8,
    settings=SerializerSettings(base=10, prec=3, signed=True, time_sep=', ', bit_sep='', minus_sign='-', half_bin_correction=False)
)

glm_hypers = dict(
    alpha=0.95,
    basic=True,
    temp=1.0,
    top_p=0.8,
    settings=SerializerSettings(base=10, prec=3, signed=True, time_sep=', ', bit_sep='', minus_sign='-', half_bin_correction=False)
)

qwen_hypers = dict(
    alpha=0.95,
    basic=True,
    temp=1.0,
    top_p=0.8,
    settings=SerializerSettings(base=10, prec=3, signed=True, time_sep=', ', bit_sep='', minus_sign='-', half_bin_correction=False)
)

moonshot_hypers = dict(
    alpha=0.95,
    basic=True,
    temp=1.0,
    top_p=0.8,
    settings=SerializerSettings(base=10, prec=3, signed=True, time_sep=', ', bit_sep='', minus_sign='-', half_bin_correction=False)
)

deepseek_hypers = dict(
    alpha=0.95,
    basic=True,
    temp=1.0,
    top_p=0.8,
    settings=SerializerSettings(base=10, prec=3, signed=True, time_sep=', ', bit_sep='', minus_sign='-', half_bin_correction=False)
)

doubao_hypers = dict(
    alpha=0.95,
    basic=True,
    temp=1.0,
    top_p=0.8,
    settings=SerializerSettings(base=10, prec=3, signed=True, time_sep=', ', bit_sep='', minus_sign='-', half_bin_correction=False)
)

qwen3_hypers = dict(
    alpha=0.95,
    basic=True,
    temp=1.0,
    top_p=0.8,
    settings=SerializerSettings(base=10, prec=3, signed=True, time_sep=', ', bit_sep='', minus_sign='-', half_bin_correction=False)
)

model_hypers = {
    'gemini-1.5-flash-latest': {'model': 'gemini-1.5-flash-latest', **gemini_hypers},
    'gemini-1.5-flash-8b': {'model': 'gemini-1.5-flash-8b', **gemini_hypers},
    'claude-3-5-haiku-20241022': {'model': 'claude-3-5-haiku-20241022', **claude_hypers},
    'claude-3-5-sonnet-20240620': {'model': 'claude-3-5-sonnet-20240620', **claude_hypers},
    "claude-3-opus-20240229": {'model': 'claude-3-opus-20240229', **claude_hypers}, 
    'glm-4-air': {'model': 'glm-4-air', **glm_hypers},
    'glm-4-long': {'model': 'glm-4-long', **glm_hypers},
    'qwen-plus': {'model': 'qwen-plus', **qwen_hypers},
    'qwen-turbo': {'model': 'qwen-turbo', **qwen_hypers},
    'moonshot-v1-8k': {'model': 'moonshot-v1-8k', **moonshot_hypers},
    'moonshot-v1-32k': {'model': 'moonshot-v1-32k', **moonshot_hypers},
    'deepseek-chat': {'model': 'deepseek-chat', **deepseek_hypers},
    "doubao-pro-4k": {'model': 'doubao-pro-4k', **doubao_hypers},
    "doubao-pro-32k": {'model': 'doubao-pro-32k', **doubao_hypers},
    "doubao-pro-128k": {'model': 'doubao-pro-128k', **doubao_hypers},
    "doubao-pro-256k": {'model': 'doubao-pro-256k', **doubao_hypers},
    "qwen3-1.7b": {'model': 'qwen3-1.7b', **qwen3_hypers},
    "qwen3-4b": {'model': 'qwen3-4b', **qwen3_hypers},    
    "qwen3-8b": {'model': 'qwen3-8b', **qwen3_hypers},  
    "qwen3-32b": {'model': 'qwen3-32b', **qwen3_hypers},    
}


model_predict_fns = {
    # 'gemini-1.5-flash-latest': get_llmtime_predictions_data,
    # 'gemini-1.5-flash-8b': get_llmtime_predictions_data,
    # 'claude-3-5-haiku-20241022': get_llmtime_predictions_data,
    # 'claude-3-5-sonnet-20240620': get_llmtime_predictions_data,
    # "claude-3-opus-20240229": get_llmtime_predictions_data, 
    # 'glm-4-air': get_llmtime_predictions_data,
    # 'glm-4-long': get_llmtime_predictions_data,
    # 'qwen-plus': get_llmtime_predictions_data,
    # 'qwen-turbo': get_llmtime_predictions_data,
    # 'moonshot-v1-8k': get_llmtime_predictions_data,
    # 'moonshot-v1-32k': get_llmtime_predictions_data,
    # 'deepseek-chat': get_llmtime_predictions_data,
    # "doubao-pro-4k": get_llmtime_predictions_data,
    # "doubao-pro-32k": get_llmtime_predictions_data,
    # "doubao-pro-128k": get_llmtime_predictions_data,
    # "doubao-pro-256k": get_llmtime_predictions_data,
    # "qwen3-1.7b": get_llmtime_predictions_data,
    # "qwen3-4b": get_llmtime_predictions_data,    
    # "qwen3-8b": get_llmtime_predictions_data,    
    "qwen3-32b": get_llmtime_predictions_data,    
}


model_names = list(model_predict_fns.keys())

## Memorization

In [ ]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
from docx import Document
from docx.shared import Inches
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error
import plotly.graph_objects as go
import mpld3
from data.small_context import get_memorization_datasets,memorization_split
from models.validation_likelihood_tuning import get_autotuned_predictions_data
import traceback
from contextlib import redirect_stdout
import sys
from data.small_context import add_noise
max_iterations = 30

noise_levels = [0.001, 0.005, 0.01, 0.02, 0.05]

datasets_noisy = {nl: memorization_split(noise=True, noise_level=nl) for nl in noise_levels}
dataset_original = memorization_split(noise=False) 

val_out_noisy = {nl: {} for nl in noise_levels}
temp_val={nl: {} for nl in noise_levels}
val_metrics_noisy = {nl: {} for nl in noise_levels}
out_noisy={}
temp_noisy={}
metrics_noisy={}
out_original = {}
temp_original={}
metrics_original = {}

output_dir = "result/memorization"
os.makedirs(output_dir, exist_ok=True)

for ds_name in dataset_original.keys():
    for model in model_names:
        log_filename = f"2_log_{model}_{ds_name}.txt"
        original_stdout = sys.stdout
        log_path = os.path.join(output_dir, log_filename)
        log_file = open(log_path, 'w', encoding='utf-8')
        sys.stdout = log_file
        print(f"--- model={model}, dataset={ds_name} ---")
        doc = Document()
        doc.add_heading('model evaluation', 0)
        doc.add_heading(f'dataset: {ds_name}', level=1)
        doc.add_heading(f'model: {model}', level=2)
        iteration = 0
        best_nmse = float('inf')  # Initialize with positive infinity (any first result will be better)
        best_metrics = None
        best_output = None
        model_hypers[model].update({'dataset_name': ds_name})
        hypers = list(grid_iter(model_hypers[model]))
        num_samples = 10

        train_original,val_original,test_original = dataset_original[ds_name]
        train_original=pd.concat([train_original, val_original])
        out_original[ds_name] = {}
        temp_original[ds_name] = {}
        metrics_original[ds_name] = {}
        while True:
            try:
                pred_dict_original = get_autotuned_predictions_data(train_original, test_original, hypers, num_samples, model_predict_fns[model], verbose=False, parallel=False)
                temp_original[ds_name][model] = pred_dict_original  
            except Exception as e:
                traceback.print_exc()
                print(f"Error with original data for dataset {ds_name} and model {model}: {e}")
                continue

            try:
                true_values_original = dataset_original[ds_name][1]
                pred_values_original = temp_original[ds_name][model]['quantile_mean']

                mse_original = temp_original[ds_name][model]['quantile_mean_mse']
                mae_original = temp_original[ds_name][model]['quantile_mean_mae']
                
                # Calculate NMSE and NMAE
                var_true = np.var(true_values_original)
                nmse_original = temp_original[ds_name][model]['quantile_mean_nmse']
                nmae_original = temp_original[ds_name][model]['quantile_mean_nmae']
                if nmse_original < best_nmse:
                    best_nmse = nmse_original
                    best_metrics = {
                        'nmse': nmse_original, 
                        'nmae': nmae_original, 
                        'mse': mse_original, 
                        'mae': mae_original
                    }
                    best_output = pred_dict_original
                    print(f"🔄update: NMSE = {best_nmse:.4f}")
                
                if nmse_original<0.4:
                    metrics_original[ds_name][model] = best_metrics
                    out_original[ds_name][model] = best_output
                    print('original data finish',nmse_original)
                    break
                else:
                    iteration += 1
                    print('original data NMSE not met, re-predicting',nmse_original)
                    if  iteration >= 5:
                        print(f'⚠️ has reached the maximum number of iterations 5, stopping prediction.')
                        metrics_original[ds_name][model] = best_metrics
                        out_original[ds_name][model] = best_output
                        print(f'⭐ save the best result NMSE={best_nmse:.4f}')
                        break
            except KeyError as e:
                traceback.print_exc()
                print(f"Key error in original predictions for dataset {ds_name} and model {model}: {e}")
                continue

        print(f"-------------------------------------------------------")
        print(f"✅ original data prediction successful",best_metrics)
        print(f"-------------------------------------------------------") 

        # 处理有噪声的数据集
        for nl in noise_levels:
            iteration = 0
            best_nmse_val = float('inf')  
            best_metrics_val = None
            best_output_val = None
            data_noisy = datasets_noisy[nl][ds_name]
            train_noisy,val_noisy,test_noisy = data_noisy

            val_out_noisy[nl][ds_name] = {}
            temp_val[nl][ds_name] = {}
            val_metrics_noisy[nl][ds_name] = {}

            out_noisy[ds_name] = {}
            temp_noisy[ds_name]={}
            metrics_noisy[ds_name] = {}

            print(f"---select noise level---")
            while True:
                try:
                    pred_dict_val = get_autotuned_predictions_data(
                        train_noisy, 
                        val_noisy,
                        hypers, num_samples, model_predict_fns[model], verbose=False, parallel=False
                    )
                    temp_val[nl][ds_name][model] = pred_dict_val
                except Exception as e:
                    traceback.print_exc()
                    print(f"Error with noisy data (noise level {nl}) for dataset {ds_name} and model {model}: {e}")
                    continue

                try:
                    true_values_val = datasets_noisy[nl][ds_name][1]
                    pred_values_val = temp_val[nl][ds_name][model]['quantile_mean']

                    mse_noisy_val = temp_val[nl][ds_name][model]['quantile_mean_mse']
                    mae_noisy_val = temp_val[nl][ds_name][model]['quantile_mean_mae']
                            
                    # Calculate NMSE and NMAE
                    var_true = np.var(true_values_val)
                    nmse_noisy_val = temp_val[nl][ds_name][model]['quantile_mean_nmse']
                    nmae_noisy_val = temp_val[nl][ds_name][model]['quantile_mean_nmae']
                    
                    if nmse_noisy_val < best_nmse_val:
                        best_nmse_val = nmse_noisy_val
                        best_metrics_val = {'nmse': nmse_noisy_val, 'nmae': nmae_noisy_val, 'mse': mse_noisy_val, 'mae': mae_noisy_val}
                        best_output_val = pred_dict_val
                        print(f"🔄Update: NMSE = {best_nmse_val:.4f}")
                    
                    if nmse_noisy_val<0.4:
                        val_metrics_noisy[nl][ds_name][model] = best_metrics_val
                        val_out_noisy[nl][ds_name][model] = best_output_val
                        print('val data finished',nmse_noisy_val)
                        break
                    else:
                        iteration += 1
                        print('val NMSE not met, re-predicting',nmse_noisy_val)
                        if  iteration >= 5:
                            print(f'⚠️ has reached the maximum number of iterations 5, stopping prediction.')
                            val_metrics_noisy[nl][ds_name][model] = best_metrics_val
                            val_out_noisy[nl][ds_name][model] = best_output_val
                            print(f'⭐ save the best result NMSE={best_nmse_val:.4f}')
                            break
                except KeyError as e:
                    traceback.print_exc()
                    print(f"Key error in noisy predictions (noise level {nl}) for dataset {ds_name} and model {model}: {e}")
                    continue

        print(f"-------------------------------------------------------")
        print(f"✅ val data prediction successful",best_metrics_val)
        print(f"-------------------------------------------------------")

        best_noise_level = None
        min_mae = float('inf')
        print("🚀 evaluate MAE...")
        for nl in val_out_noisy.keys(): 
            try:
                
                current_mae = val_out_noisy[nl][ds_name][model]['quantile_mean_mae']
                
                print(f" Noise Level {nl}: MAE = {current_mae:.5f}")
                if current_mae < min_mae:
                    min_mae = current_mae
                    best_noise_level = nl
                    
            except KeyError as e:
                traceback.print_exc()
                print(f"⚠️ Warning: can not find MAE for noise level {nl} (Key Error: {e}),skipping this noise level.")
                continue

        print(f"-------------------------------------------------------")
        print(f"✅ Best noise level (based on validation MAE): {best_noise_level} (minimum MAE: {min_mae:.4f})")
        print(f"-------------------------------------------------------")  

    
        print(f"--- Stage 2: Evaluating Dataset {ds_name} on Test Set ---")

        data = datasets_noisy[best_noise_level][ds_name]
        _,_,test_noisy = data
        train_noisy=add_noise(train_original,best_noise_level)
        iteration = 0
        best_nmse_noisy = float('inf')  
        best_metrics_noisy = None
        best_output_noisy = None
        while True:
            try:
                pred_dict_noisy = get_autotuned_predictions_data(train_noisy, test_noisy, hypers, num_samples, model_predict_fns[model], verbose=False, parallel=False)
                temp_noisy[ds_name][model] = pred_dict_noisy  
            except Exception as e:
                traceback.print_exc()
                print(f"Error with noisy data (noise level {best_noise_level}) for dataset {ds_name} and model {model}: {e}")
                continue

            try:
                true_values_noisy = datasets_noisy[best_noise_level][ds_name][2]
                pred_values_noisy = temp_noisy[ds_name][model]['quantile_mean']

                mse_noisy = temp_noisy[ds_name][model]['quantile_mean_mse']
                mae_noisy = temp_noisy[ds_name][model]['quantile_mean_mae']
                    
                # Calculate NMSE and NMAE
                var_true = np.var(true_values_noisy)
                nmse_noisy = temp_noisy[ds_name][model]['quantile_mean_nmse']
                nmae_noisy = temp_noisy[ds_name][model]['quantile_mean_nmae']
                if nmse_noisy < best_nmse_noisy:
                    best_nmse_noisy = nmse_noisy
                    best_metrics_noisy ={'nmse': nmse_noisy, 'nmae': nmae_noisy, 'mse': mse_noisy, 'mae': mae_noisy}
                    best_output_noisy = pred_dict_noisy
                    print(f"🔄 Update: NMSE = {best_nmse_noisy:.4f}")
                
                if nmse_noisy<0.4:
                    metrics_noisy[ds_name][model] = best_metrics_noisy
                    out_noisy[ds_name][model] = best_output
                    print('Test data prediction completed',nmse_noisy)
                    break
                else:
                    iteration += 1
                    print('Test data NMSE not met, re-predicting',nmse_original)
                    if  iteration >= max_iterations:
                        print(f'⚠️ have reached maximum iterations {max_iterations}, stopping prediction.')
                        metrics_noisy[ds_name][model] = best_metrics_noisy
                        out_noisy[ds_name][model] = best_output_noisy
                        print(f'⭐ Test data has saved the best prediction results, NMSE: {best_nmse_noisy:.4f}')
                        break
                #metrics_noisy[ds_name][model] = {'nmse': nmse_noisy, 'nmae': nmae_noisy, 'mse': mse_noisy, 'mae': mae_noisy}
            except KeyError as e:
                traceback.print_exc()
                print(f"Key error in noisy predictions (noise level {best_noise_level}) for dataset {ds_name} and model {model}: {e}")
                continue
        
        print(f"-------------------------------------------------------")
        print(f"✅ Test data prediction completed successfully",best_metrics_noisy)
        print(f"-------------------------------------------------------")
        

        print(f"---Result Saving---")
        # Add original data results
        if ds_name in metrics_original and model in metrics_original[ds_name]:
            doc.add_paragraph(f'Original: NMSE = {metrics_original[ds_name][model]["nmse"]}, NMAE = {metrics_original[ds_name][model]["nmae"]}, MSE = {metrics_original[ds_name][model]["mse"]}, MAE = {metrics_original[ds_name][model]["mae"]}')

        # Add val data results
        for nl in noise_levels:
            if ds_name in val_metrics_noisy[nl] and model in val_metrics_noisy[nl][ds_name]:
                doc.add_paragraph(f'Noisy (level {nl}): NMSE = {val_metrics_noisy[nl][ds_name][model]["nmse"]}, NMAE = {val_metrics_noisy[nl][ds_name][model]["nmae"]}, MSE = {val_metrics_noisy[nl][ds_name][model]["mse"]}, MAE = {val_metrics_noisy[nl][ds_name][model]["mae"]}')
        
        doc.add_paragraph(f'{ds_name} with {model} under optimal noise level (based on validation MAE): {best_noise_level} (minimum MAE: {min_mae:.4f})')
        
        # Add noisy data results
        if ds_name in metrics_noisy and model in metrics_noisy[ds_name]:
            doc.add_paragraph(f'Noisy (level {best_noise_level}): NMSE = {metrics_noisy[ds_name][model]["nmse"]}, NMAE = {metrics_noisy[ds_name][model]["nmae"]}, MSE = {metrics_noisy[ds_name][model]["mse"]}, MAE = {metrics_noisy[ds_name][model]["mae"]}')

        excel_path = os.path.join(output_dir, f'2_predictions_samples_{ds_name}_{model}.xlsx')
        with pd.ExcelWriter(excel_path) as writer:
            pd.DataFrame(out_original[ds_name][model]['raw_samples']).to_excel(writer, sheet_name='Original Samples')
            pd.DataFrame(dataset_original[ds_name][1]).to_excel(writer, sheet_name='Original test dataset')
            
            pd.DataFrame(out_noisy[ds_name][model]['raw_samples']).to_excel(writer, sheet_name='Test Samples')
            
            for nl in noise_levels:
                pd.DataFrame(datasets_noisy[nl][ds_name][0]).to_excel(writer, sheet_name=f'train dataset{nl}')
                pd.DataFrame(val_out_noisy[nl][ds_name][model]['raw_samples']).to_excel(writer, sheet_name=f'Val Samples {nl}')
                pd.DataFrame(datasets_noisy[nl][ds_name][1]).to_excel(writer, sheet_name='val dataset')

            pd.DataFrame(train_noisy).to_excel(writer, sheet_name='Noisy Train dataset')
            pd.DataFrame(datasets_noisy[best_noise_level][ds_name][2]).to_excel(writer, sheet_name='test dataset')
     
        doc_path = os.path.join(output_dir, f'2_model_predictions_results_{ds_name}_{model}.docx')
        doc.save(doc_path)
        log_file.close()
        sys.stdout = original_stdout